In [1]:
# Cell 1 — Imports & Load Data
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv("C:\\Users\\bhand\\Downloads\\e88186124ec611f1\\dataset\\train.csv")
test = pd.read_csv("C:\\Users\\bhand\\Downloads\\e88186124ec611f1\\dataset\\test.csv")
sample = pd.read_csv("C:\\Users\\bhand\\Downloads\\e88186124ec611f1\\dataset\\sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())

Train shape: (77299, 11)
Test shape: (41778, 10)
   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  


In [2]:
# Cell 2 — Handle Missing Values

# Fill RoadType nulls with mode
train['RoadType'] = train['RoadType'].fillna(train['RoadType'].mode()[0])
test['RoadType'] = test['RoadType'].fillna(train['RoadType'].mode()[0])

# Fill Temperature nulls with median
train['Temperature'] = train['Temperature'].fillna(train['Temperature'].median())
test['Temperature'] = test['Temperature'].fillna(train['Temperature'].median())

# Fill Weather nulls with mode
train['Weather'] = train['Weather'].fillna(train['Weather'].mode()[0])
test['Weather'] = test['Weather'].fillna(train['Weather'].mode()[0])

print("Train nulls after cleaning:")
print(train.isnull().sum())
print("\nTest nulls after cleaning:")
print(test.isnull().sum())

Train nulls after cleaning:
Index            0
geohash          0
day              0
timestamp        0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64

Test nulls after cleaning:
Index            0
geohash          0
day              0
timestamp        0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64


In [6]:
# Cell 3 — Encode Categorical Columns

from sklearn.preprocessing import LabelEncoder

cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

le = LabelEncoder()
for col in cat_cols:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

print("Encoding done!")
print(train[cat_cols].head())

Encoding done!
   RoadType  LargeVehicles  Landmarks  Weather
0         1              1          0        3
1         1              0          1        3
2         1              1          0        3
3         1              1          0        1
4         1              1          0        1


In [7]:
# Cell 4 — Timestamp Feature Engineering

def parse_timestamp(ts):
    parts = ts.strip().split(':')
    hour = int(parts[0])
    minute = int(parts[1])
    return hour, minute

train['hour'] = train['timestamp'].apply(lambda x: parse_timestamp(x)[0])
train['minute'] = train['timestamp'].apply(lambda x: parse_timestamp(x)[1])
train['time_in_minutes'] = train['hour'] * 60 + train['minute']

test['hour'] = test['timestamp'].apply(lambda x: parse_timestamp(x)[0])
test['minute'] = test['timestamp'].apply(lambda x: parse_timestamp(x)[1])
test['time_in_minutes'] = test['hour'] * 60 + test['minute']

# Is it rush hour? (7-9am and 5-7pm)
train['is_rush_hour'] = train['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (17 <= x <= 19) else 0)
test['is_rush_hour'] = test['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (17 <= x <= 19) else 0)

# Is it night? (10pm - 6am)
train['is_night'] = train['hour'].apply(lambda x: 1 if x >= 22 or x <= 6 else 0)
test['is_night'] = test['hour'].apply(lambda x: 1 if x >= 22 or x <= 6 else 0)

print("Time features done!")
print(train[['timestamp', 'hour', 'minute', 'time_in_minutes', 'is_rush_hour', 'is_night']].head(10))

Time features done!
  timestamp  hour  minute  time_in_minutes  is_rush_hour  is_night
0       0:0     0       0                0             0         1
1       0:0     0       0                0             0         1
2       0:0     0       0                0             0         1
3       0:0     0       0                0             0         1
4       0:0     0       0                0             0         1
5       0:0     0       0                0             0         1
6       0:0     0       0                0             0         1
7       0:0     0       0                0             0         1
8       0:0     0       0                0             0         1
9       0:0     0       0                0             0         1


In [9]:
# Cell 5 — Geohash Decoding
# Install if needed: pip install pygeohash

import subprocess
subprocess.run(['pip', 'install', 'pygeohash', '-q'])

import pygeohash as pgh

def decode_geohash(gh):
    try:
        lat, lon = pgh.decode(gh)
        return float(lat), float(lon)
    except:
        return 0.0, 0.0

train['lat'] = train['geohash'].apply(lambda x: decode_geohash(x)[0])
train['lon'] = train['geohash'].apply(lambda x: decode_geohash(x)[1])

test['lat'] = test['geohash'].apply(lambda x: decode_geohash(x)[0])
test['lon'] = test['geohash'].apply(lambda x: decode_geohash(x)[1])

print("Geohash decoded!")
print(train[['geohash', 'lat', 'lon']].head())

Geohash decoded!
  geohash       lat        lon
0  qp02z1 -5.484924  90.664673
1  qp02zt -5.462952  90.686646
2  qp08bj -5.462952  90.708618
3  qp08gt -5.462952  90.862427
4  qp02zq -5.457458  90.675659


In [10]:
# Cell 6 — Location demand aggregates (from train only)

# Average demand per geohash location
geo_mean = train.groupby('geohash')['demand'].mean().reset_index()
geo_mean.columns = ['geohash', 'geo_mean_demand']

geo_max = train.groupby('geohash')['demand'].max().reset_index()
geo_max.columns = ['geohash', 'geo_max_demand']

train = train.merge(geo_mean, on='geohash', how='left')
train = train.merge(geo_max, on='geohash', how='left')

test = test.merge(geo_mean, on='geohash', how='left')
test = test.merge(geo_max, on='geohash', how='left')

# Fill any unseen geohashes in test with overall mean
overall_mean = train['demand'].mean()
test['geo_mean_demand'] = test['geo_mean_demand'].fillna(overall_mean)
test['geo_max_demand'] = test['geo_max_demand'].fillna(train['demand'].max())

# Average demand per hour
hour_mean = train.groupby('hour')['demand'].mean().reset_index()
hour_mean.columns = ['hour', 'hour_mean_demand']

train = train.merge(hour_mean, on='hour', how='left')
test = test.merge(hour_mean, on='hour', how='left')

print("Aggregate features done!")
print(train[['geohash', 'geo_mean_demand', 'geo_max_demand', 'hour_mean_demand']].head())

Aggregate features done!
  geohash  geo_mean_demand  geo_max_demand  hour_mean_demand
0  qp02z1         0.040048        0.116299          0.083369
1  qp02zt         0.208766        0.424569          0.083369
2  qp08bj         0.127931        0.332333          0.083369
3  qp08gt         0.014381        0.037525          0.083369
4  qp02zq         0.029300        0.116787          0.083369


In [11]:
# Cell 7 — Define Features & Split

feature_cols = [
    'day', 'hour', 'minute', 'time_in_minutes',
    'is_rush_hour', 'is_night',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather',
    'lat', 'lon',
    'geo_mean_demand', 'geo_max_demand', 'hour_mean_demand'
]

X = train[feature_cols]
y = train['demand']
X_test = test[feature_cols]

print("Features shape:", X.shape)
print("Test shape:", X_test.shape)
print("Features used:", feature_cols)

Features shape: (77299, 17)
Test shape: (41778, 17)
Features used: ['day', 'hour', 'minute', 'time_in_minutes', 'is_rush_hour', 'is_night', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'lat', 'lon', 'geo_mean_demand', 'geo_max_demand', 'hour_mean_demand']


In [12]:
# Cell 8 — Train LightGBM

import subprocess
subprocess.run(['pip', 'install', 'lightgbm', '-q'])

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'n_estimators': 1000
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

    fold_score = r2_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold+1} R² Score: {fold_score:.4f}")

overall_score = max(0, 100 * r2_score(y, oof_preds))
print(f"\n✅ Overall CV Score (competition metric): {overall_score:.2f} / 100")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.0314697
[200]	valid_0's rmse: 0.0302735
[300]	valid_0's rmse: 0.0297398
[400]	valid_0's rmse: 0.0295535
[500]	valid_0's rmse: 0.029345
[600]	valid_0's rmse: 0.0292632
[700]	valid_0's rmse: 0.0291921
[800]	valid_0's rmse: 0.0291146
[900]	valid_0's rmse: 0.0290587
[1000]	valid_0's rmse: 0.0290474
Did not meet early stopping. Best iteration is:
[961]	valid_0's rmse: 0.0290454
Fold 1 R² Score: 0.9583
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.0317919
[200]	valid_0's rmse: 0.0305141
[300]	valid_0's rmse: 0.0301514
[400]	valid_0's rmse: 0.0300131
[500]	valid_0's rmse: 0.0299095
[600]	valid_0's rmse: 0.0298443
Early stopping, best iteration is:
[585]	valid_0's rmse: 0.0298353
Fold 2 R² Score: 0.9577
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.0313799
[200]	valid_0's rmse: 0.0300503
[300]	valid_0's rmse: 0.0296335
[400]	valid_0'

In [13]:
# Cell 9 — Feature Importance

import pandas as pd

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df)

             feature  importance
10       Temperature       16745
14   geo_mean_demand       14462
15    geo_max_demand       12050
12               lat       11716
3    time_in_minutes       11604
13               lon       10630
16  hour_mean_demand        5246
1               hour        5044
2             minute        3659
7      NumberofLanes        3039
11           Weather        1577
6           RoadType        1069
9          Landmarks        1029
0                day         903
4       is_rush_hour         560
8      LargeVehicles         510
5           is_night         327


In [15]:
# Cell 10 — Generate Submission

# Clip predictions to valid range (demand is between 0 and 1)
test_preds_clipped = np.clip(test_preds, 0, 1)

# Create submission dataframe
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds_clipped
})

# Verify shape should be 41778 x 2
print("Submission shape:", submission.shape)
print("\nSample predictions:")
print(submission.head(10))
print("\nPrediction stats:")
print(submission['demand'].describe())

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("\n✅ submission.csv saved successfully!")

Submission shape: (41778, 2)

Sample predictions:
   Index    demand
0      0  0.051428
1      1  0.032883
2      2  0.030281
3      3  0.034697
4      4  0.054101
5      5  0.008086
6      6  0.039512
7      7  0.080460
8      8  0.040659
9      9  0.055529

Prediction stats:
count    41778.000000
mean         0.125507
std          0.168584
min          0.000187
25%          0.028215
50%          0.062827
75%          0.134319
max          1.000000
Name: demand, dtype: float64

✅ submission.csv saved successfully!


In [16]:
# Cell 11 — Verify submission matches requirements

sub_check = pd.read_csv('submission.csv')
print("Shape check (should be 41778 x 2):", sub_check.shape)
print("Columns check (should be Index, demand):", list(sub_check.columns))
print("Any nulls?", sub_check.isnull().sum().sum())
print("All Index values present?", len(sub_check) == len(test))
print("\n✅ Ready to submit!")

Shape check (should be 41778 x 2): (41778, 2)
Columns check (should be Index, demand): ['Index', 'demand']
Any nulls? 0
All Index values present? True

✅ Ready to submit!
